# Notebook 03 — Phase 1: ShuffleNetV2 0.5× Liveness Baseline

**Phase:** 1 (Real-but-weak models, final piece)  
**Purpose:** Train a binary live-vs-spoof classifier on CelebA-Spoof and ship it as `shufflenet_liveness.tflite`. Replaces the last remaining dummy (`shufflenet_dummy.tflite`) in `mobile_app/assets/models/`.  
**Expected runtime:** ~15–25 minutes  
**GPU required:** **YES — T4 ×2 strongly recommended**

## What this notebook does

1. Auto-discovers where the CelebA-Spoof dataset is mounted (handles both standard and `datasets/<user>/` mount patterns).
2. Samples **10k live + 10k spoof** images for training, **2k + 2k** for validation. Train/val are split by **subject** so we measure generalization, not memorization.
3. Builds a `tf.data` pipeline with on-RAM caching, random horizontal flip + brightness + contrast augmentation.
4. Builds **ShuffleNetV2 0.5×** from scratch (~80 lines, TFLite-compatible).
5. Trains for `EPOCHS` epochs (default 10) with Adam.
6. Converts the trained Keras model → `shufflenet_liveness.tflite` FP32.
7. Validates the TFLite output shape against `shared_contracts/`.
8. Saves training curves + history JSON.

## What to send back to Claude

Paste the **literal output of cells 5, 7, 11, 13, 15, 17, 19** (discovery, sampling, model summary, training, conversion, validation, final summary). If anything fails, paste the **full traceback** of the failing cell.

## What to do after the run

Download from Kaggle Output:
- `/kaggle/working/models/shufflenet_liveness.tflite` → `mobile_app/assets/models/`
- `/kaggle/working/reports/shufflenet_training_history.json` → `ml_pipeline/evaluation/reports/`
- `/kaggle/working/reports/shufflenet_training_curves.png` → `ml_pipeline/evaluation/reports/`

Then commit; ledger entry; delete `shufflenet_dummy.tflite`.

## 1 — Config

Tune these knobs if you want a shorter/longer run.

In [ ]:
# Tunable knobs
SAMPLES_PER_CLASS_TRAIN = 10_000   # 10k live + 10k spoof in training
SAMPLES_PER_CLASS_VAL   = 2_000    # 2k live + 2k spoof in validation
MAX_PER_SUBJECT         = 6        # cap per subject for diversity
IMAGE_SIZE              = (112, 112)
BATCH_SIZE              = 64
EPOCHS                  = 10
LR                      = 1e-3
SEED                    = 42

# Fixed paths
WORK_DIR    = "/kaggle/working"
MODELS_OUT  = "/kaggle/working/models"
REPORTS_OUT = "/kaggle/working/reports"

import os
os.makedirs(MODELS_OUT, exist_ok=True)
os.makedirs(REPORTS_OUT, exist_ok=True)

print("Config locked.")

## 2 — Clone repo + discover dataset mount

Kaggle mount paths for CelebA-Spoof can take either form depending on how the dataset is attached. We try both, then fall back to a shallow search.

In [ ]:
import subprocess, os, sys, glob, random, json

# Clone repo
REPO_URL = "https://github.com/MalayM09/nhai.git"
REPO_DIR = os.path.join(WORK_DIR, "nhai")
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

# Discover CelebA-Spoof Data root
print("\n/kaggle/input contents:")
if os.path.isdir("/kaggle/input"):
    for d in sorted(os.listdir("/kaggle/input")):
        print(f"  {d}")

GUESS_PATHS = [
    "/kaggle/input/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof/Data",
    "/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof/Data",
    "/kaggle/input/celeba-spoof-for-face-antispoofing/CelebA_Spoof/Data",
]
DATA_ROOT = None
for guess in GUESS_PATHS:
    if os.path.isdir(os.path.join(guess, "train")):
        DATA_ROOT = guess
        break

if DATA_ROOT is None:
    # Slow fallback: walk one level under /kaggle/input/ for a CelebA_Spoof dir
    for top in os.listdir("/kaggle/input"):
        hits = glob.glob(f"/kaggle/input/{top}/**/CelebA_Spoof/Data/train", recursive=True)
        if hits:
            DATA_ROOT = hits[0].rsplit("/train", 1)[0]
            break

assert DATA_ROOT, "Could not locate CelebA_Spoof/Data — paste /kaggle/input/ tree to Claude"
print(f"\nDATA_ROOT: {DATA_ROOT}")
print(f"Subdirs:   {sorted(os.listdir(DATA_ROOT))}")

TRAIN_ROOT = os.path.join(DATA_ROOT, "train")
TEST_ROOT  = os.path.join(DATA_ROOT, "test")
print(f"\nTRAIN subjects: {len(os.listdir(TRAIN_ROOT))}")
print(f"TEST  subjects: {len(os.listdir(TEST_ROOT))}")

## 3 — Sample file lists

Walk subject directories, taking up to `MAX_PER_SUBJECT` images per class per subject, until we hit the quotas. Caches the file lists in `/kaggle/working/` so reruns are fast.

In [ ]:
import numpy as np

CACHE = os.path.join(WORK_DIR, "celeba_spoof_filelist.npz")
IMG_EXTS = (".jpg", ".jpeg", ".png")

def list_imgs(d):
    if not os.path.isdir(d):
        return []
    return [os.path.join(d, f) for f in os.listdir(d)
            if f.lower().endswith(IMG_EXTS)]

def sample_split(root, n_per_class, max_per_subject, seed):
    rng = random.Random(seed)
    subjects = os.listdir(root)
    rng.shuffle(subjects)
    live, spoof = [], []
    for sid in subjects:
        if len(live) >= n_per_class and len(spoof) >= n_per_class:
            break
        subj_dir = os.path.join(root, sid)
        if len(live) < n_per_class:
            imgs = list_imgs(os.path.join(subj_dir, "live"))
            rng.shuffle(imgs)
            live.extend(imgs[:max_per_subject])
        if len(spoof) < n_per_class:
            imgs = list_imgs(os.path.join(subj_dir, "spoof"))
            rng.shuffle(imgs)
            spoof.extend(imgs[:max_per_subject])
    live, spoof = live[:n_per_class], spoof[:n_per_class]
    files = live + spoof
    labels = [0] * len(live) + [1] * len(spoof)   # 0 = live, 1 = spoof  (per contract)
    return files, labels

if os.path.isfile(CACHE):
    print(f"Loading cached file lists from {CACHE}")
    z = np.load(CACHE, allow_pickle=True)
    train_files = z["train_files"].tolist();  train_labels = z["train_labels"].tolist()
    val_files   = z["val_files"].tolist();    val_labels   = z["val_labels"].tolist()
else:
    print("First run — walking subject directories…")
    train_files, train_labels = sample_split(TRAIN_ROOT, SAMPLES_PER_CLASS_TRAIN, MAX_PER_SUBJECT, SEED)
    val_files,   val_labels   = sample_split(TEST_ROOT,  SAMPLES_PER_CLASS_VAL,   MAX_PER_SUBJECT, SEED + 1)
    np.savez(CACHE,
             train_files=np.array(train_files, dtype=object),
             train_labels=np.array(train_labels, dtype=np.int32),
             val_files=np.array(val_files, dtype=object),
             val_labels=np.array(val_labels, dtype=np.int32))
    print(f"Cached file lists to {CACHE}")

from collections import Counter
print(f"\nTrain: {len(train_files):>6d}  class counts {dict(Counter(train_labels))}")
print(f"Val:   {len(val_files):>6d}  class counts {dict(Counter(val_labels))}")
print(f"\nExamples:")
for i in [0, len(train_files)//2 - 1, len(train_files) - 1]:
    print(f"  [label={train_labels[i]}] {train_files[i]}")

## 4 — `tf.data` pipeline

- **Pre-shuffle the file list Python-side** before `from_tensor_slices`. This is the critical fix: `sample_split` returns files in order `[live × 10k] + [spoof × 10k]`, and the in-pipeline shuffle buffer (8k) can't mix across the 10k class boundary inside `.cache()` — so every batch ended up class-pure, model learned to predict whichever class was current, val accuracy stuck at 50%.
- Decode → resize 112×112 → normalize `[0, 1]` (matches `shared_contracts/`).
- `.cache()` after decode so the network mount is hit **once**, not every epoch.
- In-pipeline shuffle with `buffer_size=8192` for per-epoch perturbation on top of the pre-shuffled order.
- Augment: random hflip + brightness + contrast (no spatial crop — already cropped via resize).

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def decode_resize(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0   # contract: [0, 1]
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_ds(files, labels, training):
    # CRITICAL: pre-shuffle (files, labels) Python-side so cache() stores a
    # mixed order. Without this the cached pipeline emits class-pure batches
    # (live block then spoof block) and the model fails to generalize.
    combined = list(zip(files, labels))
    random.Random(SEED + (0 if training else 1)).shuffle(combined)
    files, labels = [list(x) for x in zip(*combined)]

    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    if training:
        ds = ds.shuffle(buffer_size=8192, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_files, train_labels, training=True)
val_ds   = make_ds(val_files,   val_labels,   training=False)

# Smoke-test one batch — class balance check matters here too
for x, y in train_ds.take(1):
    counts = {0: int(tf.reduce_sum(tf.cast(y == 0, tf.int32))),
              1: int(tf.reduce_sum(tf.cast(y == 1, tf.int32)))}
    print(f"batch x: shape={x.shape} dtype={x.dtype} range=[{tf.reduce_min(x):.3f}, {tf.reduce_max(x):.3f}]")
    print(f"batch y: shape={y.shape} dtype={y.dtype} class counts={counts}  (both should be > 0)")
print(f"\nGPU(s) visible to TF: {tf.config.list_physical_devices('GPU') or 'NONE — switch accelerator to T4 ×2'}")

## 5 — ShuffleNetV2 0.5× (inline)

Width multiplier `0.5×`: stage channels `[48, 96, 192]`, blocks per stage `[4, 8, 4]`. All tensor ops in the Functional API graph go through the **Keras 3 `keras.ops.*` namespace**, not raw `tf.*` — Keras 3 forbids calling raw TF functions on `KerasTensor` placeholders during model construction. The `ChannelShuffle` layer is a `Layer` subclass that uses `keras.ops.reshape` + `keras.ops.transpose` and converts cleanly to TFLite.

In [ ]:
from tensorflow.keras import layers as L
import keras  # Keras 3 namespace — use keras.ops.* for tensor ops on KerasTensors

class ChannelShuffle(tf.keras.layers.Layer):
    def __init__(self, groups=2, **kwargs):
        super().__init__(**kwargs)
        self.groups = groups
    def call(self, x):
        h, w, c = x.shape[1], x.shape[2], x.shape[3]
        # -1 for batch dim works across symbolic and eager modes
        x = keras.ops.reshape(x, (-1, h, w, self.groups, c // self.groups))
        x = keras.ops.transpose(x, (0, 1, 2, 4, 3))
        x = keras.ops.reshape(x, (-1, h, w, c))
        return x
    def get_config(self):
        return {**super().get_config(), "groups": self.groups}

def _conv_bn_relu(x, ch, k=1, s=1):
    x = L.Conv2D(ch, k, strides=s, padding="same", use_bias=False)(x)
    x = L.BatchNormalization()(x)
    return L.ReLU()(x)

def _dw_bn(x, k=3, s=1):
    x = L.DepthwiseConv2D(k, strides=s, padding="same", use_bias=False)(x)
    return L.BatchNormalization()(x)

def shufflev2_block(x, out_ch, stride):
    if stride == 1:
        c = x.shape[-1]
        # keras.ops.split returns a list; unpack into the two halves
        x1, x2 = keras.ops.split(x, indices_or_sections=2, axis=-1)
        m = _conv_bn_relu(x2, c // 2, k=1)
        m = _dw_bn(m, k=3, s=1)
        m = _conv_bn_relu(m, c // 2, k=1)
        out = L.Concatenate()([x1, m])
    else:
        # Downsample branch (left)
        l = _dw_bn(x, k=3, s=2)
        l = _conv_bn_relu(l, out_ch // 2, k=1)
        # Main branch (right)
        r = _conv_bn_relu(x, out_ch // 2, k=1)
        r = _dw_bn(r, k=3, s=2)
        r = _conv_bn_relu(r, out_ch // 2, k=1)
        out = L.Concatenate()([l, r])
    return ChannelShuffle(groups=2)(out)

def build_shufflenet_v2_050(input_shape=(112, 112, 3), num_classes=2):
    inp = L.Input(shape=input_shape, name="image")
    x = _conv_bn_relu(inp, 24, k=3, s=2)
    x = L.MaxPooling2D(3, strides=2, padding="same")(x)
    for ch, reps in zip([48, 96, 192], [4, 8, 4]):
        x = shufflev2_block(x, ch, stride=2)
        for _ in range(reps - 1):
            x = shufflev2_block(x, ch, stride=1)
    x = _conv_bn_relu(x, 1024, k=1, s=1)
    x = L.GlobalAveragePooling2D()(x)
    out = L.Dense(num_classes, activation="softmax", name="liveness")(x)
    return tf.keras.Model(inp, out, name="shufflenet_v2_050_liveness")

model = build_shufflenet_v2_050(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
model.summary()
n_params = model.count_params()
print(f"\nTotal params: {n_params:,}  (~{n_params * 4 / 1024 / 1024:.2f} MB FP32, ~{n_params / 1024 / 1024:.2f} MB INT8 estimate)")

## 6 — Train

Adam optimizer, sparse categorical crossentropy. Metric: **accuracy only** during training (AUC is computed once at the end on the full val set).

Why no inline AUC: Keras 3's `tf.keras.metrics.AUC` calls `UnsortedSegmentSum` internally, which doesn't reconcile shapes correctly under Kaggle's T4 ×2 mirrored strategy — the per-replica batch (64) and global batch (128) mismatch. We avoid the trap entirely by computing AUC post-hoc with `sklearn`.

In [ ]:
from sklearn.metrics import roc_auc_score

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=2,
)

# Save Keras checkpoint
KERAS_PATH = os.path.join(WORK_DIR, "shufflenet_liveness.keras")
model.save(KERAS_PATH)
print(f"\nKeras model saved -> {KERAS_PATH}  ({os.path.getsize(KERAS_PATH)/1024/1024:.2f} MB)")

# Compute final AUCs post-hoc with sklearn (avoids the multi-GPU AUC metric issue)
print("\nComputing final AUC on validation set...")
val_probs  = model.predict(val_ds, verbose=0)[:, 1]                      # P(spoof)
val_labels = np.concatenate([y.numpy() for _, y in val_ds], axis=0)
final_val_auc = float(roc_auc_score(val_labels, val_probs))
print(f"Final val AUC: {final_val_auc:.4f}")

print("Computing final AUC on a training subset...")
train_probs_list, train_labels_list = [], []
for xb, yb in train_ds.take(100):                                        # ~6.4k samples is plenty
    train_probs_list.append(model(xb, training=False).numpy()[:, 1])
    train_labels_list.append(yb.numpy())
train_probs  = np.concatenate(train_probs_list)
train_labels = np.concatenate(train_labels_list)
final_train_auc = float(roc_auc_score(train_labels, train_probs))
print(f"Final train AUC (6.4k subset): {final_train_auc:.4f}")

# Persist history JSON with the final AUCs appended
HIST_PATH = os.path.join(REPORTS_OUT, "shufflenet_training_history.json")
hist_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
hist_data["final_val_auc"]          = final_val_auc
hist_data["final_train_auc_subset"] = final_train_auc
with open(HIST_PATH, "w") as f:
    json.dump(hist_data, f, indent=2)
print(f"\nHistory saved -> {HIST_PATH}")

## 7 — Convert to TFLite (FP32 — INT8 PTQ comes in Notebook 05)

In [ ]:
TFLITE_PATH = os.path.join(MODELS_OUT, "shufflenet_liveness.tflite")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_bytes)
print(f"TFLite saved -> {TFLITE_PATH}  ({os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB)")

## 8 — Validate TFLite output against the contract

Load through `tf.lite.Interpreter`, run a real validation batch, confirm:
1. Input shape `[1, 112, 112, 3]`.
2. Output shape `[1, 2]` (softmax).
3. Probabilities sum to 1.
4. The model's predictions roughly track Keras predictions on the same images (sanity).

In [ ]:
it = tf.lite.Interpreter(model_path=TFLITE_PATH)
it.allocate_tensors()
inp_d = it.get_input_details()[0]
out_d = it.get_output_details()[0]
print(f"Input  {inp_d['name']}: shape={inp_d['shape'].tolist()} dtype={inp_d['dtype'].__name__}")
print(f"Output {out_d['name']}: shape={out_d['shape'].tolist()} dtype={out_d['dtype'].__name__}")

assert inp_d['shape'].tolist() == [1, 112, 112, 3], "input shape mismatch"
assert out_d['shape'].tolist() == [1, 2], "output shape mismatch"

# Run TFLite on one val batch image, compare to Keras
for xb, yb in val_ds.take(1):
    keras_probs = model(xb).numpy()      # [B, 2]
    tfl_probs   = np.zeros_like(keras_probs)
    for i in range(xb.shape[0]):
        it.set_tensor(inp_d['index'], xb[i:i+1].numpy().astype(inp_d['dtype']))
        it.invoke()
        tfl_probs[i] = it.get_tensor(out_d['index'])[0]
    # Mean absolute disagreement (TFLite vs Keras)
    diff = float(np.mean(np.abs(keras_probs - tfl_probs)))
    keras_acc = float(np.mean(np.argmax(keras_probs, 1) == yb.numpy()))
    tfl_acc   = float(np.mean(np.argmax(tfl_probs,  1) == yb.numpy()))
    print(f"\nOne-batch comparison (batch_size={xb.shape[0]}):")
    print(f"  Keras accuracy:        {keras_acc:.4f}")
    print(f"  TFLite accuracy:       {tfl_acc:.4f}")
    print(f"  Mean |keras - tflite|: {diff:.6f}  (should be < 1e-4 for FP32)")
    print(f"  Softmax sum check:     {float(np.mean(np.sum(tfl_probs, axis=1))):.6f}  (should be ~1.0)")
    break

## 9 — Training curves + final summary

In [ ]:
import matplotlib.pyplot as plt

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (key, ylabel) in zip(axes, [("loss", "Loss"), ("accuracy", "Accuracy")]):
    ax.plot(h[key],         label=f"train {key}")
    ax.plot(h[f"val_{key}"], label=f"val {key}")
    ax.set_xlabel("epoch"); ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle(f"ShuffleNetV2 0.5× — CelebA-Spoof binary liveness   (final val AUC = {final_val_auc:.4f})")
fig.tight_layout()
PLOT_PATH = os.path.join(REPORTS_OUT, "shufflenet_training_curves.png")
fig.savefig(PLOT_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"\nPlot saved -> {PLOT_PATH}")

print("\n" + "=" * 78)
print("Final summary")
print("=" * 78)
print(f"Train samples:               {len(train_files):,}  (per-class: {SAMPLES_PER_CLASS_TRAIN:,})")
print(f"Val   samples:               {len(val_files):,}  (per-class: {SAMPLES_PER_CLASS_VAL:,})")
print(f"Epochs:                      {EPOCHS}")
print(f"Final train acc:             {h['accuracy'][-1]:.4f}")
print(f"Final val   acc:             {h['val_accuracy'][-1]:.4f}")
print(f"Final train AUC (subset):    {final_train_auc:.4f}")
print(f"Final val   AUC:             {final_val_auc:.4f}")
print(f"\nTFLite file:                 {TFLITE_PATH}")
print(f"TFLite size:                 {os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB")
print(f"Keras checkpoint:            {KERAS_PATH}  ({os.path.getsize(KERAS_PATH)/1024/1024:.2f} MB)")
print(f"History JSON:                {HIST_PATH}")
print(f"Curves PNG:                  {PLOT_PATH}")

## What to send back to Claude

Paste the literal output of these cells (in this order):

1. **Cell 5** — discovery (`DATA_ROOT`, train/test subject counts)
2. **Cell 7** — sampling (class counts, example file paths)
3. **Cell 11** — model summary (param count + size estimates)
4. **Cell 13** — training (per-epoch loss/acc/auc lines)
5. **Cell 15** — TFLite size after conversion
6. **Cell 17** — TFLite validation (Keras vs TFLite agreement, accuracy)
7. **Cell 19** — final summary table

If anything fails, paste the **full traceback**.

## What to do after the run

```bash
# Download from Kaggle Output panel or via CLI
kaggle kernels output <your-username>/03-phase1-shufflenet-liveness -p /tmp/kaggle_out

# Move into the repo
cd ~/Desktop/nhai
mv /tmp/kaggle_out/models/shufflenet_liveness.tflite mobile_app/assets/models/
mkdir -p ml_pipeline/evaluation/reports
mv /tmp/kaggle_out/reports/shufflenet_training_history.json ml_pipeline/evaluation/reports/
mv /tmp/kaggle_out/reports/shufflenet_training_curves.png   ml_pipeline/evaluation/reports/

# Drop the last dummy
git rm mobile_app/assets/models/shufflenet_dummy.tflite

git add mobile_app/assets/models/shufflenet_liveness.tflite ml_pipeline/evaluation/reports/
git commit -m "ml(phase1): ship real ShuffleNetV2 0.5× liveness .tflite (CelebA-Spoof, FP32)"
git push
```

Tell teammate: "Phase 1 model bundle complete — point liveness loader at `shufflenet_liveness.tflite`. All four models are real now."